## Test Notebook: Bronze Ingestion (COPY INTO)

Component test for `02_bronze_copy_into.sql`: validates interaction between Volume CSV files and Delta table.

**Prerequisites:**
- Volume path `/Volumes/dev_automotive/landing/landing_raw/` exists.
- Schema `dev_automotive.bronze` exists.

**Tests:** Record count, schema/parsing, `_rescued_data` for malformed rows, idempotency.

### 1. Setup & Mock Data Generation

In [ ]:
volume_path = "/Volumes/dev_automotive/landing/landing_raw"
test_good_file = f"{volume_path}/test_unit_photo.csv"
test_bad_file = f"{volume_path}/test_unit_photo_malformed.csv"

# 1. Perfect CSV (matches expected schema: _c0, photo_url, id)
dbutils.fs.put(test_good_file, """_c0,photo_url,id
101,http://img.com/1.jpg,500.5
102,http://img.com/2.jpg,600.5""", overwrite=True)

# 2. Malformed CSV: row 2 has extra column "oops" → should land in _rescued_data
dbutils.fs.put(test_bad_file, """_c0,photo_url,id
201,http://img.com/3.jpg,700.5
202,http://img.com/4.jpg,800.5,oops_extra_column""", overwrite=True)

print(f"Created mock files at: {volume_path}")

### 2. Execution (Run SQL)

In [ ]:
# Reset table for clean test
spark.sql("DROP TABLE IF EXISTS dev_automotive.bronze.photo_raw")

# Re-create table (matches 02_bronze_copy_into.sql DDL)
spark.sql("""
CREATE TABLE IF NOT EXISTS dev_automotive.bronze.photo_raw (
  photo_seq_id INT,
  photo_url STRING,
  id DOUBLE,
  load_dt TIMESTAMP,
  source STRING,
  _rescued_data STRING
) USING DELTA
""")

# COPY INTO from GOOD file
spark.sql(f"""
COPY INTO dev_automotive.bronze.photo_raw
FROM (
  SELECT
    CAST(_c0 AS INT) AS photo_seq_id,
    photo_url,
    id,
    current_timestamp() AS load_dt,
    _metadata.file_path AS source,
    _rescued_data
  FROM 'file:{test_good_file}'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'rescuedDataColumn' = '_rescued_data')
""")

# COPY INTO from BAD file (extra column → _rescued_data)
spark.sql(f"""
COPY INTO dev_automotive.bronze.photo_raw
FROM (
  SELECT
    CAST(_c0 AS INT) AS photo_seq_id,
    photo_url,
    id,
    current_timestamp() AS load_dt,
    _metadata.file_path AS source,
    _rescued_data
  FROM 'file:{test_bad_file}'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'rescuedDataColumn' = '_rescued_data')
""")

print("COPY INTO completed for good and bad files.")

### 3. Assertions (Tests)

---

**Critical note (SQL vs CSV header):**  
The pipeline uses `_c0` in the SELECT because the test and production CSVs use `_c0` as the first column header. If your real CSV uses a different header (e.g. `SequenceID` or no header), change the SELECT to use the **actual** column name from the file. With `'header' = 'true'`, the reader uses the CSV header; with `'header' = 'false'`, the first column is typically named `_c0`.

In [ ]:
df = spark.table("dev_automotive.bronze.photo_raw")

# --- TEST 1: Record count ---
total_count = df.count()
assert total_count == 4, f"TEST FAILED: Expected 4 rows, found {total_count}"
print("✅ Test 1 Passed: Total Record Count")

# --- TEST 2: Schema / parsing (good data) ---
good_record = df.filter("photo_seq_id = 101").first()
assert good_record["id"] == 500.5, "TEST FAILED: Data type mismatch or parsing error on ID"
assert good_record["_rescued_data"] is None or good_record["_rescued_data"] == "", "TEST FAILED: Good record was falsely rescued"
print("✅ Test 2 Passed: Data Parsing")

# --- TEST 3: Rescue logic (bad data) ---
bad_record = df.filter("photo_seq_id = 202").first()
assert bad_record["_rescued_data"] is not None, "TEST FAILED: Malformed row was NOT rescued"
assert "oops_extra_column" in (bad_record["_rescued_data"] or ""), "TEST FAILED: Rescued data content missing"
print("✅ Test 3 Passed: Rescue Logic")

# --- TEST 4: Idempotency ---
spark.sql(f"""
COPY INTO dev_automotive.bronze.photo_raw
FROM (
  SELECT
    CAST(_c0 AS INT) AS photo_seq_id,
    photo_url,
    id,
    current_timestamp() AS load_dt,
    _metadata.file_path AS source,
    _rescued_data
  FROM 'file:{test_good_file}'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'rescuedDataColumn' = '_rescued_data')
""")

new_count = spark.table("dev_automotive.bronze.photo_raw").count()
assert new_count == 4, f"TEST FAILED: Idempotency check failed. Rows increased to {new_count}"
print("✅ Test 4 Passed: Idempotency")

### 4. Teardown (Cleanup)

In [ ]:
# Remove test files from volume
dbutils.fs.rm(test_good_file)
dbutils.fs.rm(test_bad_file)

# Optional: drop test table to leave env clean (uncomment if desired)
# spark.sql("DROP TABLE IF EXISTS dev_automotive.bronze.photo_raw")

print("🧹 Cleanup complete.")